# Step 0 — 資料健檢

確認資料完整、欄位一致、無異常缺漏。
此步驟無需調整參數。

In [1]:
# ===== 共用參數 =====
exec(open('../config/params.py').read())
# ==============================

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import yaml


In [3]:
df = pd.read_csv(DATA_PATH)
df['order_created_at'] = pd.to_datetime(df['order_created_at'], format=DATETIME_FORMAT)

print("=" * 60)
print("資料健檢報告")
print("=" * 60)

# 1. 基本資訊
print(f"\n✓ 總筆數: {len(df):,}")
print(f"✓ 欄位數: {len(df.columns)}")
print(f"✓ 欄位: {list(df.columns)}")

# 2. Null 檢查
nulls = df.isnull().sum()
nulls_pct = (nulls / len(df) * 100).round(1)
null_report = pd.DataFrame({'null_count': nulls, 'null_pct': nulls_pct})
null_report = null_report[null_report['null_count'] > 0]
if len(null_report) > 0:
    print(f"\n⚠️  有 null 的欄位:")
    for col, row in null_report.iterrows():
        print(f"   {col}: {row['null_count']:,} ({row['null_pct']}%)")
else:
    print(f"\n✓ 無 null 欄位")

# 3. order_id 唯一性
dup = df['order_id'].duplicated().sum()
print(f"\n{'✓' if dup == 0 else '✗'} order_id 重複: {dup}")

# 4. Device 數量
print(f"✓ Unique devices: {df['device_id'].nunique()}")

# 5. 時間範圍
print(f"✓ 時間範圍: {df['order_created_at'].min()} ~ {df['order_created_at'].max()}")
print(f"  天數: {(df['order_created_at'].max() - df['order_created_at'].min()).days}")

# 6. 異常值檢查
checks = {
    'total_duration_seconds <= 0': (df['total_duration_seconds'] <= 0).sum(),
    'file_count <= 0': (df['file_count'] <= 0).sum(),
}
for col in df.select_dtypes(include='number').columns:
    if 'duration' in col or 'seconds' in col or 'minutes' in col:
        neg = (df[col] < 0).sum()
        if neg > 0:
            checks[f'{col} < 0'] = neg

for desc, count in checks.items():
    print(f"{'✓' if count == 0 else '✗'} {desc}: {count}")


資料健檢報告

✓ 總筆數: 30,000
✓ 欄位數: 23
✓ 欄位: ['device_id', 'device_mode_name', 'order_created_at', 'order_id', 'file_count', 'loc_1', 'loc_2', 'system_name', 'queue_duration_seconds', 'per_file_duration_avg_seconds', 'per_file_duration_max_seconds', 'per_file_duration_p95_seconds', 'device_duration_avg_seconds', 'device_duration_max_seconds', 'device_duration_p95_seconds', 'db_duration_avg_seconds', 'db_duration_max_seconds', 'db_duration_p95_seconds', 'inner_processing_duration_avg_seconds', 'inner_processing_duration_max_seconds', 'inner_processing_duration_p95_seconds', 'total_file_duration_minutes', 'total_duration_seconds']

⚠️  有 null 的欄位:
   device_mode_name: 4,990.0 (16.6%)
   loc_2: 7,951.0 (26.5%)

✓ order_id 重複: 0
✓ Unique devices: 2000
✓ 時間範圍: 2026-02-28 21:27:25.526546 ~ 2026-03-31 20:31:56.707658
  天數: 30
✓ total_duration_seconds <= 0: 0
✓ file_count <= 0: 0


In [4]:
# 0. Schema 驗證
with open('../config/schema.yaml') as f:
    schema = yaml.safe_load(f)
expected_cols = [field['name'] for field in schema['fields']]
actual_cols = list(df.columns)
missing = set(expected_cols) - set(actual_cols)
extra = set(actual_cols) - set(expected_cols)

print("=" * 60)
print("Schema 驗證")
print("=" * 60)
if missing:
    print(f"\n✗ 缺少欄位: {sorted(missing)}")
else:
    print(f"\n✓ 所有 schema 欄位都存在 ({len(expected_cols)} 個)")
if extra:
    print(f"ℹ️  額外欄位（不在 schema 中）: {sorted(extra)}")

# Check nullable
for field in schema['fields']:
    col = field['name']
    if col in df.columns and not field.get('nullable', False):
        n = df[col].isnull().sum()
        if n > 0:
            print(f"✗ {col}: 有 {n} 筆 null，但 schema 定義為 non-nullable")

Schema 驗證

✓ 所有 schema 欄位都存在 (23 個)


In [5]:
# 7. 數值欄位 describe
print("\n=== 數值欄位統計 ===")
print(df.describe().round(1).to_string())



=== 數值欄位統計 ===
                    order_created_at  file_count  queue_duration_seconds  per_file_duration_avg_seconds  per_file_duration_max_seconds  per_file_duration_p95_seconds  device_duration_avg_seconds  device_duration_max_seconds  device_duration_p95_seconds  db_duration_avg_seconds  db_duration_max_seconds  db_duration_p95_seconds  inner_processing_duration_avg_seconds  inner_processing_duration_max_seconds  inner_processing_duration_p95_seconds  total_file_duration_minutes  total_duration_seconds
count                          30000     30000.0                 30000.0                        30000.0                        30000.0                        30000.0                      30000.0                      30000.0                      30000.0                  30000.0                  30000.0                  30000.0                                30000.0                                30000.0                                30000.0                      30000.0             

In [6]:
# 8. 關鍵 percentiles（記下來，後續步驟會用到）
print("\n=== 關鍵 Percentiles（供後續步驟參考）===")
for col in ['file_count', 'queue_duration_seconds', 'device_duration_avg_seconds',
            'db_duration_avg_seconds', 'total_duration_seconds']:
    pcts = df[col].quantile([0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).round(1)
    print(f"\n{col}:")
    print(f"  P25={pcts[0.25]}, P50={pcts[0.5]}, P75={pcts[0.75]}, "
          f"P90={pcts[0.9]}, P95={pcts[0.95]}, P99={pcts[0.99]}, max={df[col].max()}")



=== 關鍵 Percentiles（供後續步驟參考）===

file_count:
  P25=38.0, P50=99.0, P75=258.0, P90=609.0, P95=1029.0, P99=2721.0, max=41092

queue_duration_seconds:
  P25=0.0, P50=0.0, P75=1.6, P90=3.8, P95=4.9, P99=27.1, max=85.8

device_duration_avg_seconds:
  P25=1.2, P50=2.1, P75=3.9, P90=6.8, P95=9.7, P99=22.8, max=504.35

db_duration_avg_seconds:
  P25=0.0, P50=0.0, P75=0.1, P90=0.2, P95=0.3, P99=3.8, max=1198.31

total_duration_seconds:
  P25=36.1, P50=86.0, P75=232.6, P90=627.8, P95=1241.8, P99=7049.5, max=1042566.5


In [7]:
# 9. 前 5 筆
print("\n=== 前 5 筆資料 ===")
print(df.head().to_string())



=== 前 5 筆資料 ===
  device_id device_mode_name           order_created_at    order_id  file_count  loc_1   loc_2 system_name  queue_duration_seconds  per_file_duration_avg_seconds  per_file_duration_max_seconds  per_file_duration_p95_seconds  device_duration_avg_seconds  device_duration_max_seconds  device_duration_p95_seconds  db_duration_avg_seconds  db_duration_max_seconds  db_duration_p95_seconds  inner_processing_duration_avg_seconds  inner_processing_duration_max_seconds  inner_processing_duration_p95_seconds  total_file_duration_minutes  total_duration_seconds
0  DEV-0624          MDL-022 2026-03-20 05:52:40.533355  ORD-000000          41  FAB-C     NaN   SYS-ALPHA                     0.0                           6.32                          26.16                          17.58                         3.32                        18.00                         9.82                     0.00                     0.50                     0.10                                   3.00   

In [8]:
# Export summary to reports/
import io, sys
_buf = io.StringIO()
_old_stdout = sys.stdout
sys.stdout = _buf

print("=" * 60)
print("Step 0 — 資料健檢報告")
print("=" * 60)
print(f"\n總筆數: {len(df):,}  （分析的訂單總數）")
print(f"欄位數: {len(df.columns)}  （CSV 的欄位數量）")
print(f"Unique devices: {df['device_id'].nunique()}  （不重複的設備數量）")
print(f"時間範圍: {df['order_created_at'].min()} ~ {df['order_created_at'].max()}")
print(f"天數: {(df['order_created_at'].max() - df['order_created_at'].min()).days}  （資料涵蓋天數）")

nulls = df.isnull().sum()
null_cols = nulls[nulls > 0]
if len(null_cols) > 0:
    print(f"\nNull 欄位:  （有空值的欄位，device_mode_name 和 loc_2 允許為空）")
    for col, n in null_cols.items():
        print(f"  {col}: {n:,} ({100*n/len(df):.1f}%)")

print(f"\n關鍵 Percentiles:  （P50=中位數, P95=95%訂單在此值以下, P99=99%以下）")
for col in ['file_count', 'queue_duration_seconds', 'device_duration_avg_seconds',
            'db_duration_avg_seconds', 'total_duration_seconds']:
    pcts = df[col].quantile([0.5, 0.75, 0.95, 0.99]).round(1)
    desc = {'file_count': '每筆訂單的檔案數',
            'queue_duration_seconds': '訂單在 queue 等待的秒數',
            'device_duration_avg_seconds': '單檔 device command 平均秒數',
            'db_duration_avg_seconds': '單檔 DB 查詢平均秒數',
            'total_duration_seconds': '訂單從建立到完成的總秒數'}[col]
    print(f"  {col}  （{desc}）")
    print(f"    P50={pcts[0.5]}, P75={pcts[0.75]}, P95={pcts[0.95]}, P99={pcts[0.99]}, max={df[col].max()}")

sys.stdout = _old_stdout
with open(str(REPORTS_DIR / 'step0_summary.txt'), 'w') as f:
    f.write(_buf.getvalue())
print(f"Saved: reports/step0_summary.txt")


Saved: reports/step0_summary.txt
